# QuantOpt: Institutional Portfolio Optimization

This notebook demonstrates the capabilities of the `quantopt` library. We will:
1. Generate synthetic price data using Geometric Brownian Motion (GBM)
2. Estimate expected returns and covariance (using PCA Factor Model)
3. Optimize portfolios (Max Sharpe, Min Vol, CVaR, Risk Parity)
4. Plot the Efficient Frontier and Risk metrics
5. Run a Walk-Forward backtest with transaction costs
6. Generate a Tear Sheet of performance analytics

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

import quantopt as qo
from quantopt.plotting import charts

ModuleNotFoundError: No module named 'quantopt'

## 1. Synthetic Data Generation

In [ ]:
np.random.seed(42)

n_assets = 10
n_days = 1260 # 5 years

# Base parameters
mu_true = np.random.uniform(0.02, 0.15, n_assets)
vol_true = np.random.uniform(0.10, 0.30, n_assets)

# Generate a PSD correlation matrix
factor_loadings = np.random.normal(0, 1, (n_assets, 3))
cov_true = factor_loadings @ factor_loadings.T
D = np.diag(1 / np.sqrt(np.diag(cov_true)))
corr_true = D @ cov_true @ D
cov_matrix = np.outer(vol_true, vol_true) * corr_true

# Simulate daily log returns
dt = 1/252
drift = (mu_true - 0.5 * vol_true**2) * dt
L = np.linalg.cholesky(cov_matrix * dt)

Z = np.random.standard_normal((n_days, n_assets))
log_ret = drift + Z @ L.T

# Prices
dates = pd.date_range("2018-01-01", periods=n_days, freq="B")
prices = pd.DataFrame(
    100 * np.exp(np.cumsum(log_ret, axis=0)), 
    index=dates, 
    columns=[f"Asset_{i+1}" for i in range(n_assets)]
)

# Plot prices
prices.plot(figsize=(10, 5), title="Synthetic Asset Prices", legend=False)
plt.show()

## 2. Estimation (Expected Returns & Covariance)

In [ ]:
returns = qo.prices_to_returns(prices)

# Estimate historical mean returns (with slight winsorization for robustness)
mu_estimator = qo.MeanHistoricalReturn(returns)
mu = mu_estimator.estimate()

# Estimate covariance using a statistical PCA Factor Model (3 factors)
cov_estimator = qo.FactorModelCovariance(returns, n_components=3)
Sigma = cov_estimator.estimate()

print("Expected Returns (Ann.):")
print(mu.round(4))

charts.plot_correlation_matrix(cov_estimator, title="Factor Model Correlation Matrix")
plt.show()

## 3. Optimization & The Efficient Frontier

In [ ]:
ef = qo.EfficientFrontier(mu, Sigma, l2_gamma=0.01)

# 1. Max Sharpe
w_sharpe = ef.max_sharpe()
ef.clean_weights() # Cleans internal weights and normalizes
ret_shp, vol_shp, sr_shp = ef.portfolio_performance(mu, Sigma)
print(f"Max Sharpe: Ret {ret_shp:.2%}, Vol {vol_shp:.2%}, SR {sr_shp:.2f}")

# 2. Min Volatility
w_minvol = ef.min_volatility()
ret_mvol, vol_mvol, sr_mvol = ef.portfolio_performance(mu, Sigma)
print(f"Min Vol:    Ret {ret_mvol:.2%}, Vol {vol_mvol:.2%}, SR {sr_mvol:.2f}")

# 3. Compute the full Efficient Frontier for plotting
frontier_df = ef.efficient_frontier_points(n_points=40)

# Plot
benchmarks = {
    "Equal Weight": (qo.SampleCovariance(returns).estimate().values.diagonal().mean()**0.5, mu.mean())
}
charts.plot_efficient_frontier(
    frontier_df, 
    tangency_point=(vol_shp, ret_shp), 
    benchmark_points=benchmarks,
    title="Mean-Variance Efficient Frontier"
)
plt.show()

charts.plot_weights(ef.weights_, title="Max Sharpe Portfolio Weights")
plt.show()

## 4. Alternative Risk Frameworks: CVaR and Risk Parity

In [ ]:
# Equal Risk Contribution (Risk Parity)
rp = qo.RiskParity(Sigma)
w_rp = rp.optimize()

charts.plot_risk_contributions(w_rp, Sigma, title="Risk Parity: Equal Risk Contribution")
plt.show()

# Conditional Value-at-Risk (CVaR) Optimization
# Minimize 95% CVaR using historical scenarios
cvar_opt = qo.CVaROptimizer(returns, beta=0.95)
w_cvar = cvar_opt.optimize()

print(f"Optimized CVaR (95%): {cvar_opt.cvar_:.2%}")
charts.plot_weights(w_cvar, title="CVaR Optimized Portfolio Weights")
plt.show()

## 5. Walk-Forward Backtesting

In [ ]:
# Define a backtest configuration with reasonable transaction costs (10bps spread)
config = qo.BacktestConfig(
    lookback_days=252*2,       # 2 years of history for rolling estimation
    rebalance_freq="M",        # Monthly rebalancing
    transaction_cost=qo.TransactionCostModel(proportional=0.0010),
    max_turnover=0.30          # Limit one-way turnover to 30% per rebalance
)

# Define two strategies using the OptimizerFactory
factory = qo.OptimizerFactory()

def strategy_risk_parity(window_returns):
    # Estimate rolling covariance
    S = qo.LedoitWolfCovariance(window_returns).estimate()
    return factory.build("risk_parity", Sigma=S)

def strategy_min_vol(window_returns):
    # Estimate rolling mu and covariance
    m = qo.MeanHistoricalReturn(window_returns).estimate()
    S = qo.FactorModelCovariance(window_returns).estimate()
    return factory.build("min_vol", mu=m, Sigma=S, l2_gamma=0.01)

strategies = {
    "Risk Parity": strategy_risk_parity,
    "Min Volatility": strategy_min_vol
}

# Equal weight benchmark
ew_returns = returns.mean(axis=1)
ew_prices = (1 + ew_returns).cumprod() * 100

# Run engine
engine = qo.WalkForwardBacktester(
    prices=prices,
    optimizer_factory=None, # Not used in run_comparison
    config=config,
    benchmark_prices=ew_prices
)

results = engine.run_comparison(strategies, config)

# Plot Cumulative Returns
rets_dict = {
    name: res.portfolio_returns for name, res in results.items()
}
rets_dict["Equal Weight Benchmark"] = ew_returns.loc[rets_dict["Risk Parity"].index]

charts.plot_cumulative_returns(rets_dict, title="Out-of-Sample Walk-Forward Backtest")
plt.show()

## 6. Performance Tear Sheet

In [ ]:
rp_res = results["Risk Parity"]

display(rp_res.summary)

charts.plot_drawdown(rp_res.portfolio_returns, title="Risk Parity Drawdown Profile")
plt.show()

fig, ax = charts.plot_rolling_metrics(rp_res.portfolio_returns, window=126)
plt.show()

charts.plot_weights_history(rp_res.weights_history, title="Rolling Target Weights")
plt.show()